# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All dataset elements are referenced by their unique `@id` identifiers as mandated by the Croissant specification.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show basic metadata
print("Dataset Name:", metadata.name)
print("Dataset Description:", metadata.description)
print("Dataset Identifier:", metadata.identifier)
print("Dataset Version:", metadata.version)
print("Date Published:", metadata.datePublished)
print("Keywords:", metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their respective `@id`s. All entities in the dataset are referenced by their `@id`.

We will list the record sets defined in the dataset schema, along with their fields and columns.

In [ ]:
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet Name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Field Name: {field.name}")
        print(f"    Field @id: {field.id}")
        print(f"    Data Type: {field.data_type}")
        if hasattr(field, 'column'):
            print(f"    Column @id: {field.column.id}")
        print("")
    print("----\n")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis.
All loading operations reference the record set and field `@id` values from the overview above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print("RecordSet @ids to be loaded:")
print(record_set_ids)

# Load records for each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet @id: {rs_id}")
    if len(records) > 0:
        print(f"Columns for {rs_id}: {dataframes[rs_id].columns.tolist()}\n")
    else:
        print("No records found.\n")

# Display head of the first record set dataframe (if any records are available)
if len(record_set_ids) > 0 and not dataframes[record_set_ids[0]].empty:
    print(f"First few records from RecordSet @id: {record_set_ids[0]}")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data.

We'll demonstrate filtering, normalizing, and grouping using one RecordSet and its fields referenced by their `@id`s.

In [ ]:
# Choose the main record set for EDA
main_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_rs_id] if main_rs_id else pd.DataFrame()

# Find a numeric field's @id (e.g., age)
numeric_field_id = None
group_field_id = None
for field in [rs for rs in record_sets if rs.id == main_rs_id][0].fields:
    if field.data_type in ['Integer', 'Float', 'Number', 'schema:Integer', 'schema:Float'] and numeric_field_id is None:
        numeric_field_id = field.id  # Use first numeric field
    if field.data_type == 'Text' and group_field_id is None:
        group_field_id = field.id  # Use first text field

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

# Filter numeric field if present
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field @id
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Numeric field not found in dataframe.")

## 5. Visualization
Visualize distributions or relationships between fields using record set and field `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Distribution of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Numeric field not available for visualization.")

## 6. Conclusion
In this notebook, we've explored the FAIR^2 dataset using `mlcroissant`, referencing all entities via their `@id`. We loaded metadata, overviewed schema, extracted tabular data, applied filtering and normalization, and visualized key numeric and grouped attributes. This approach ensures reproducible FAIR analytics as prescribed by the Croissant schema.
